# 11 · Inferencia con diseño muestral complejo (linealización de Taylor)

Los notebooks 08-09 estiman errores estándar **robustos por conglomerado × año** (`varunit`), lo que captura la agrupación (*clustering*) del diseño CASEN pero **omite la estratificación** (`varstrat`). Un árbitro de una revista técnica exige la inferencia de diseño completo. Este notebook implementa la **varianza linealizada de Taylor** —el estándar de `svy: regress` de Stata— que incorpora estratos, conglomerados y factores de expansión simultáneamente, y verifica que las conclusiones no cambian.

La varianza linealizada del estimador ponderado \(\hat\beta = (X'WX)^{-1}X'Wy\) es

$$\widehat{V}(\hat\beta) = (X'WX)^{-1}\,\hat G\,(X'WX)^{-1},\qquad \hat G=\sum_h \frac{n_h}{n_h-1}\sum_{j\in h}(s_{hj}-\bar s_h)(s_{hj}-\bar s_h)',$$

donde \(s_{hj}=\sum_{i\in hj} w_i x_i \hat e_i\) es la suma de las contribuciones al puntaje (*score*) del conglomerado \(j\) del estrato \(h\), \(\bar s_h\) su media en el estrato y \(n_h\) el número de conglomerados del estrato. La estratificación resta la variación *entre* estratos, por lo que produce errores estándar iguales o **menores** que los de solo conglomerado: la inferencia previa del proyecto es, si acaso, conservadora.

In [1]:
import os
import numpy as np
import pandas as pd
import patsy
import statsmodels.formula.api as smf
from scipy import stats

os.makedirs('outputs/data', exist_ok=True)
RUTA_CASEN = '../../CASEN'
COLS = ['sexo','edad','e6a','o10','oficio4_08','ytrabajocor','expr','activ','varunit','varstrat']

frames = []
for anio in [2022, 2024]:
    df = pd.read_stata(f'{RUTA_CASEN}/casen_{anio}.dta', columns=COLS, convert_categoricals=False)
    df['anio'] = anio
    frames.append(df)
panel = pd.concat(frames, ignore_index=True)
m = panel[(panel['activ']==1)&(panel['ytrabajocor'].notna())&(panel['ytrabajocor']>0)&
          (panel['oficio4_08'].notna())].copy()

NIVEL_EDUC = {1:'Básica',2:'Básica',3:'Básica',4:'Básica',5:'Básica',6:'Básica',7:'Básica',
    8:'Media',9:'Media',10:'Media',11:'Media',12:'Técnica sup.',13:'Universitaria',14:'Posgrado',15:'Posgrado'}
m['mujer'] = (m['sexo']==2).astype(int)
m['nivel_grp'] = m['e6a'].map(NIVEL_EDUC)
m['log_ingreso'] = np.log(m['ytrabajocor'])
m['edad2'] = m['edad']**2
m = m[(m['o10']>0)&(m['o10']<=112)]
conteo = m['oficio4_08'].value_counts()
val = set(conteo[conteo>=30].index)
m['oficio4_grp'] = pd.Categorical(m['oficio4_08'].apply(lambda c: str(int(c)) if c in val else 'otras'))
m['ciuo_1digito'] = pd.Categorical((m['oficio4_08']//1000).astype(int).astype(str))
m['nivel_grp'] = pd.Categorical(m['nivel_grp'])
m['anio'] = pd.Categorical(m['anio'])
m = m.dropna(subset=['nivel_grp'])
# El diseno se anida dentro de cada encuesta anual: estrato y PSU se cruzan con el anio
m['estrato'] = m['anio'].astype(str) + '_' + m['varstrat'].astype(int).astype(str)
m['psu'] = m['estrato'] + '_' + m['varunit'].astype(int).astype(str)
print(f'n = {len(m):,}  |  estratos: {m["estrato"].nunique():,}  |  conglomerados: {m["psu"].nunique():,}')

n = 174,924  |  estratos: 1,511  |  conglomerados: 24,450


## 1. La función de varianza de diseño y su validación

La verificación de que la implementación es correcta: forzando un único estrato, la varianza linealizada debe reproducir exactamente el error estándar cluster-robusto de `statsmodels`.

In [2]:
def var_diseno(formula, datos, estrato, psu, peso):
    """Ajuste WLS + varianza linealizada de Taylor (estrato+PSU+peso). Devuelve
    (modelo, ee_svy: pd.Series). Con un solo estrato reproduce la varianza cluster."""
    mod = smf.wls(formula, data=datos, weights=datos[peso]).fit()
    y, X = patsy.dmatrices(formula, datos, return_type='dataframe')
    Xv = X.values; w = datos[peso].values
    beta = mod.params.reindex(X.columns).values
    e = y.values.flatten() - Xv @ beta
    u = (w * e)[:, None] * Xv                                  # score por obs (n x k)
    bread = np.linalg.inv(Xv.T @ (w[:, None] * Xv))            # (X'WX)^-1
    su = pd.DataFrame(u); su['estrato'] = estrato.values; su['psu'] = psu.values
    s_psu = su.groupby(['estrato','psu']).sum()
    k = u.shape[1]; G = np.zeros((k, k))
    for _, grp in s_psu.groupby(level=0):
        nh = len(grp)
        if nh < 2:
            continue
        D = grp.values - grp.values.mean(axis=0)
        G += (nh/(nh-1)) * (D.T @ D)
    V = bread @ G @ bread
    ee = pd.Series(np.sqrt(np.diag(V)), index=X.columns)
    return mod, ee

f_gran = 'log_ingreso ~ mujer + edad + edad2 + C(nivel_grp) + o10 + C(anio) + C(oficio4_grp)'

# Validacion: un solo estrato == cluster-robusto de statsmodels
mod_cl = smf.wls(f_gran, data=m, weights=m['expr']).fit(cov_type='cluster', cov_kwds={'groups': m['psu']})
_, ee_1estrato = var_diseno(f_gran, m, pd.Series(['all']*len(m), index=m.index), m['psu'], 'expr')
print('Validación (EE del coef. mujer):')
print(f'  statsmodels cluster : {mod_cl.bse["mujer"]:.6f}')
print(f'  linealizada 1 estrato: {ee_1estrato["mujer"]:.6f}')
print(f'  coinciden: {np.isclose(mod_cl.bse["mujer"], ee_1estrato["mujer"], atol=1e-4)}')

Validación (EE del coef. mujer):
  statsmodels cluster : 0.005548
  linealizada 1 estrato: 0.005542
  coinciden: True


## 2. Brecha ajustada bajo diseño muestral completo

Reestimamos las especificaciones centrales con la varianza de diseño (estrato + conglomerado + peso) y las comparamos con la inferencia cluster-only del resto del proyecto.

In [3]:
def brecha_ic(mod, ee, nombre):
    b = mod.params['mujer']; se = ee['mujer']
    pval = 2*(1 - stats.norm.cdf(abs(b/se)))
    return {'especificación': nombre, 'brecha_pct': (np.exp(b)-1)*100,
            'ee_log': se, 'ic95_inf': (np.exp(b-1.96*se)-1)*100,
            'ic95_sup': (np.exp(b+1.96*se)-1)*100, 'p': pval}

f_ampl = 'log_ingreso ~ mujer + edad + edad2 + C(nivel_grp) + o10 + C(anio) + C(ciuo_1digito)'
filas_svy, filas_cl = [], []
for nombre, formula in [('Ocupación 1 dígito', f_ampl), ('Ocupación 4 dígitos', f_gran)]:
    mod, ee = var_diseno(formula, m, m['estrato'], m['psu'], 'expr')
    filas_svy.append(brecha_ic(mod, ee, nombre))
    mc = smf.wls(formula, data=m, weights=m['expr']).fit(cov_type='cluster', cov_kwds={'groups': m['psu']})
    filas_cl.append(brecha_ic(mc, mc.bse, nombre))

svy = pd.DataFrame(filas_svy); cl = pd.DataFrame(filas_cl)
comp = svy[['especificación','brecha_pct']].copy()
comp['EE diseño completo'] = svy['ee_log'].round(5)
comp['EE cluster-only'] = cl['ee_log'].round(5)
comp['IC95% diseño'] = svy.apply(lambda r: f'[{r.ic95_inf:.1f}, {r.ic95_sup:.1f}]', axis=1)
comp['razón EE'] = (svy['ee_log']/cl['ee_log']).round(4)
comp['brecha_pct'] = comp['brecha_pct'].round(2)
print(comp.to_string(index=False))
comp.to_csv('outputs/data/inferencia_diseno_muestral.csv', index=False, encoding='utf-8-sig')

     especificación  brecha_pct  EE diseño completo  EE cluster-only   IC95% diseño  razón EE
 Ocupación 1 dígito      -20.86             0.00524          0.00526 [-21.7, -20.0]    0.9959
Ocupación 4 dígitos      -15.31             0.00553          0.00555 [-16.2, -14.4]    0.9970


### Interpretación

La estimación puntual es idéntica —el diseño muestral afecta la varianza, no el estimador ponderado— y los errores estándar bajo diseño completo son **prácticamente iguales, y marginalmente menores**, que los cluster-only (razón ≈ 0.997). La brecha ajustada por ocupación a 4 dígitos es **-15.3% (IC 95% [-16.2, -14.4])** bajo la inferencia de diseño correcta, indistinguible de la reportada en el resto del proyecto.

La lectura para la publicación es doble: (i) la inferencia del proyecto es robusta a la especificación completa del diseño muestral; (ii) al omitir la estratificación, los errores estándar cluster-only son levemente **conservadores** (más anchos), de modo que ningún resultado significativo del proyecto se debe a subestimar la varianza. La aparente pequeñez del efecto de la estratificación es esperable: con 756 estratos por año y ~16 conglomerados por estrato, la variación entre estratos que la estratificación remueve es una fracción menor de la varianza total.